In [2]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import gc

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("C:\\Repos\\Xakaton\\data\\raw")

pd.set_option('display.max_rows', None)      # все строки
pd.set_option('display.max_columns', None)   # все столбцы
pd.set_option('display.width', None)         # не разрывать строки по ширине консоли
pd.set_option('display.max_colwidth', None)  # не обрезать текст внутри ячеек
pd.set_option('display.expand_frame_repr', False) # не переносить широкие таблицы на новые строки

In [1]:
# ============================================================
# 0. Минимальный dfs для восстановлялки
# ============================================================

DATA_DIR = Path("C:\\Repos\\Xakaton\\data\\raw")

dfs = {}

dfs["users_courses"] = pd.read_csv(
    DATA_DIR / "users_courses.csv",
    usecols=["user_id", "course_id"]
)

dfs["user_lessons"] = pd.read_csv(
    DATA_DIR / "user_lessons.csv",
    usecols=["user_id", "lesson_id", "group_id", "users_course_id"]
)

dfs["wk_users_courses_actions"] = pd.read_csv(
    DATA_DIR / "wk_users_courses_actions.csv",
    usecols=["user_id", "users_course_id"]
)

dfs["users"] = pd.read_csv(
    DATA_DIR / "users.csv",
    usecols=["id", "type"]
)

# Карта колонок для конвертации в целочисленный тип
INT_COLS_MAP = {
    "users_courses": ["user_id", "course_id"],
    "user_lessons": ["user_id", "lesson_id", "group_id", "users_course_id"],
    'wk_users_courses_actions': ['user_id', 'users_course_id']
}

# Конвертация в точности в стиле EDA
for tbl, cols in INT_COLS_MAP.items():
    for col in cols:
        if col not in dfs[tbl].columns:
            continue
        if pd.api.types.is_numeric_dtype(dfs[tbl][col]):
            continue
        dfs[tbl][col] = pd.to_numeric(
            dfs[tbl][col].astype(str).str.replace(",", "", regex=False),
            errors="coerce"
        ).astype("Int64")

print("Loaded tables:")
for k, v in dfs.items():
    print(k, v.shape)

Loaded tables:
users_courses (290835, 2)
user_lessons (3070664, 4)
wk_users_courses_actions (12909207, 2)
users (95395, 2)


In [3]:
# ============================================================
# users: базовая очистка + исключение учителей
# ============================================================

# 1. Приводим ключ к единому имени
dfs["users"]["user_id"] = dfs["users"]["id"].copy()

# 2. Выделяем учителей
# True для teacher / agent аккаунтов
dfs["users"]["is_teacher"] = dfs["users"]["type"].astype("string").str.contains("Agent", na=False)

# 3. Сохраняем множества id до фильтрации
teacher_ids = set(dfs["users"].loc[dfs["users"]["is_teacher"], "user_id"])
pupil_ids = set(dfs["users"].loc[~dfs["users"]["is_teacher"], "user_id"])

print(f"Всего users: {len(dfs['users'])}")
print(f"Учителей: {len(teacher_ids)}")
print(f"Учеников: {len(pupil_ids)}")

# 4. Оставляем только учеников
dfs["users"] = (
    dfs["users"]
    .loc[~dfs["users"]["is_teacher"]]
    .drop(columns=["type", "is_teacher"], errors="ignore")
    .reset_index(drop=True)
)

dfs['users'] = dfs['users'].drop(columns=['id'])

display(dfs["users"].head())

Всего users: 95395
Учителей: 4748
Учеников: 90647


,user_id
0,675765
1,679040
2,679200
3,678069
4,693126


In [4]:
# ============================================================
# Удаляем teacher-аккаунты из всех таблиц
# ============================================================

def filter_out_teachers_from_all_tables(
    dfs: dict[str, pd.DataFrame],
    teacher_ids: set,
) -> pd.DataFrame:
    """
    Удаляет строки, связанные с teacher-аккаунтами, из всех таблиц,
    где встречаются user-like идентификаторы.
    """

    # user-like ключи, которые стоит проверить во всех таблицах
    candidate_user_cols = [
        "user_id",
    ]

    stats = []

    for table_name, df in dfs.items():
        before = len(df)
        filtered_df = df

        applied_cols = []

        for col in candidate_user_cols:
            if col in filtered_df.columns:
                filtered_df = filtered_df.loc[~filtered_df[col].isin(teacher_ids)].copy()
                applied_cols.append(col)

        after = len(filtered_df)
        removed = before - after

        dfs[table_name] = filtered_df

        stats.append({
            "table": table_name,
            "checked_columns": ", ".join(applied_cols) if applied_cols else "",
            "rows_before": before,
            "rows_after": after,
            "rows_removed": removed,
            "share_removed": removed / before if before > 0 else 0.0,
        })

    return pd.DataFrame(stats).sort_values("rows_removed", ascending=False).reset_index(drop=True)


teacher_filter_stats = filter_out_teachers_from_all_tables(dfs, teacher_ids)
display(teacher_filter_stats)

,table,checked_columns,rows_before,rows_after,rows_removed,share_removed
0,wk_users_courses_actions,user_id,12909207,12633684,275523,0.021343
1,user_lessons,user_id,3070664,3008595,62069,0.020214
2,users_courses,user_id,290835,267206,23629,0.081245
3,users,user_id,90647,90647,0,0.000000


In [5]:
# ============================================================
# 1. Список уникальных user_id и выбор по индексу i
# ============================================================

unique_user_ids = (
    dfs["user_lessons"]["user_id"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print("Количество уникальных user_id:", len(unique_user_ids))
display(unique_user_ids.head(20).to_frame(name="user_id"))

i = 0  # меняй вручную
target_user_id = unique_user_ids.iloc[i]

print("i =", i)
print("target_user_id =", target_user_id)

Количество уникальных user_id: 71520


,user_id
0,665768
1,665778
2,665791
3,665795
4,665800
5,665805
6,665806
7,665808
8,665810
9,665811


i = 0
target_user_id = 665768


In [6]:
# ============================================================
# 2. Первые 10 строк user_lessons для выбранного пользователя
# ============================================================

display(
    dfs["user_lessons"]
    .loc[dfs["user_lessons"]["user_id"] == target_user_id]
    .head(10)
)

,user_id,lesson_id,group_id,users_course_id
0,665768,50004620,50004743,449077
1,665768,50004625,50004748,449077
2,665768,50004621,50004744,449077
3,665768,50004622,50004745,449077
4,665768,50004623,50004746,449077


In [7]:
# ============================================================
# 3A. Берем lesson_id и group_id из j-й строки пользователя
# ============================================================

user_lessons_of_target = dfs["user_lessons"].loc[
    dfs["user_lessons"]["user_id"] == target_user_id
].copy()

display(user_lessons_of_target.head(10))

j = 0  # меняй вручную: какую строку пользователя взять
seed_row = user_lessons_of_target.iloc[j]

target_lesson_id = seed_row["lesson_id"]
target_group_id = seed_row["group_id"]

print("target_user_id  =", target_user_id)
print("target_lesson_id =", target_lesson_id)
print("target_group_id  =", target_group_id)

,user_id,lesson_id,group_id,users_course_id
0,665768,50004620,50004743,449077
1,665768,50004625,50004748,449077
2,665768,50004621,50004744,449077
3,665768,50004622,50004745,449077
4,665768,50004623,50004746,449077


target_user_id  = 665768
target_lesson_id = 50004620
target_group_id  = 50004743


In [8]:
# ============================================================
# 3B. Все пользователи с тем же lesson_id и group_id
# ============================================================

same_group_same_lesson = dfs["user_lessons"].loc[
    (dfs["user_lessons"]["lesson_id"] == target_lesson_id) &
    (dfs["user_lessons"]["group_id"] == target_group_id)
].copy()

same_group_user_ids = (
    same_group_same_lesson["user_id"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print("Все строки этой группы на этом уроке:")
display(same_group_same_lesson.head(20))

print("Первые 10 user_id из этой группы на этом уроке:")
display(same_group_user_ids.head(10).to_frame(name="user_id"))

Все строки этой группы на этом уроке:


,user_id,lesson_id,group_id,users_course_id
0,665768,50004620,50004743,449077
443,667077,50004620,50004743,451163
479,665835,50004620,50004743,449226
640,665974,50004620,50004743,449554
663,669037,50004620,50004743,455083
906,666073,50004620,50004743,449669
1036,671685,50004620,50004743,462283
1353,666573,50004620,50004743,450367
1451,666067,50004620,50004743,449663
1538,666159,50004620,50004743,449797


Первые 10 user_id из этой группы на этом уроке:


,user_id
0,665768
1,665835
2,665854
3,665867
4,665884
5,665974
6,666006
7,666007
8,666012
9,666067


In [9]:
# ============================================================
# 4. Для первых 10 найденных пользователей — их строки из users_courses
# ============================================================

first_10_group_user_ids = same_group_user_ids.head(10).tolist()

users_courses_first_10 = dfs["users_courses"].loc[
    dfs["users_courses"]["user_id"].isin(first_10_group_user_ids)
].copy()

print("Первые 10 user_id группы:")
print(first_10_group_user_ids)

print("Строки users_courses для этих пользователей:")
display(
    users_courses_first_10
    .sort_values(["user_id", "course_id"])
    .head(100)
)

Первые 10 user_id группы:
[665768, 665835, 665854, 665867, 665884, 665974, 666006, 666007, 666012, 666067]
Строки users_courses для этих пользователей:


,user_id,course_id
59156,665768,50000592
14429,665768,170000688
71011,665835,50000592
75813,665854,50000592
18603,665854,170000688
78558,665867,50000592
25515,665884,50000592
8336,665884,170000688
55585,665974,50000592
68442,666006,50000592


In [10]:
# ============================================================
# 4A. Список course_id для каждого из первых 10 пользователей
# ============================================================

courses_per_user_first_10 = (
    users_courses_first_10
    .groupby("user_id")["course_id"]
    .agg(lambda s: sorted(set(s.dropna())))
    .reset_index()
    .rename(columns={"course_id": "course_id_list"})
)

display(courses_per_user_first_10)

,user_id,course_id_list
0,665768,"[50000592, 170000688]"
1,665835,[50000592]
2,665854,"[50000592, 170000688]"
3,665867,[50000592]
4,665884,"[50000592, 170000688]"
5,665974,[50000592]
6,666006,[50000592]
7,666007,[50000592]
8,666012,[50000592]
9,666067,[50000592]


In [11]:
# ============================================================
# 5. Пересечение course_id у пользователей найденной группы
# ============================================================

group_users_courses = dfs["users_courses"].loc[
    dfs["users_courses"]["user_id"].isin(same_group_user_ids.tolist())
].copy()

courses_per_user = (
    group_users_courses
    .groupby("user_id")["course_id"]
    .agg(lambda s: set(s.dropna()))
)

if len(courses_per_user) == 0:
    course_intersection = set()
else:
    course_intersection = set.intersection(*courses_per_user.tolist())

course_intersection = sorted(course_intersection)

print("Пересечение course_id у всех пользователей данной группы на данном уроке:")
print(course_intersection)

display(pd.DataFrame({"intersection_course_id": course_intersection}))

Пересечение course_id у всех пользователей данной группы на данном уроке:
[np.int64(50000592)]


,intersection_course_id
0,50000592


In [12]:
# Проверка совпадения множеств user_id в users_courses и user_lessons

uc_user_ids = set(dfs["users_courses"]["user_id"].dropna().unique())
ul_user_ids = set(dfs["user_lessons"]["user_id"].dropna().unique())

only_in_users_courses = sorted(uc_user_ids - ul_user_ids)
only_in_user_lessons = sorted(ul_user_ids - uc_user_ids)

comparison_df = pd.DataFrame([
    {"metric": "users_courses_unique_user_id", "value": len(uc_user_ids)},
    {"metric": "user_lessons_unique_user_id", "value": len(ul_user_ids)},
    {"metric": "intersection_user_id", "value": len(uc_user_ids & ul_user_ids)},
    {"metric": "only_in_users_courses", "value": len(only_in_users_courses)},
    {"metric": "only_in_user_lessons", "value": len(only_in_user_lessons)},
    {"metric": "sets_equal", "value": uc_user_ids == ul_user_ids},
])

display(comparison_df)

print("Первые 20 user_id только в users_courses:")
display(pd.DataFrame({"user_id": only_in_users_courses[:20]}))

print("Первые 20 user_id только в user_lessons:")
display(pd.DataFrame({"user_id": only_in_user_lessons[:20]}))

,metric,value
0,users_courses_unique_user_id,84572
1,user_lessons_unique_user_id,71520
2,intersection_user_id,71488
3,only_in_users_courses,13084
4,only_in_user_lessons,32
5,sets_equal,False


Первые 20 user_id только в users_courses:


,user_id
0,665740
1,665741
2,665749
3,665750
4,665753
5,665755
6,665757
7,665758
8,665760
9,665761


Первые 20 user_id только в user_lessons:


,user_id
0,761581
1,761586
2,761588
3,761599
4,761603
5,761609
6,761614
7,761617
8,761618
9,761652


In [13]:
# Сравнение множеств уникальных users_course_id
# между user_lessons и wk_users_courses_actions

ul_uc_ids = set(dfs["user_lessons"]["users_course_id"].dropna().unique())
wka_uc_ids = set(dfs["wk_users_courses_actions"]["users_course_id"].dropna().unique())

only_in_user_lessons = sorted(ul_uc_ids - wka_uc_ids)
only_in_wk_actions = sorted(wka_uc_ids - ul_uc_ids)

comparison_users_course_id_df = pd.DataFrame([
    {"metric": "user_lessons_unique_users_course_id", "value": len(ul_uc_ids)},
    {"metric": "wk_users_courses_actions_unique_users_course_id", "value": len(wka_uc_ids)},
    {"metric": "intersection_users_course_id", "value": len(ul_uc_ids & wka_uc_ids)},
    {"metric": "only_in_user_lessons", "value": len(only_in_user_lessons)},
    {"metric": "only_in_wk_users_courses_actions", "value": len(only_in_wk_actions)},
    {"metric": "sets_equal", "value": ul_uc_ids == wka_uc_ids},
])

display(comparison_users_course_id_df)

print("Первые 20 users_course_id только в user_lessons:")
display(pd.DataFrame({"users_course_id": only_in_user_lessons[:20]}))

print("Первые 20 users_course_id только в wk_users_courses_actions:")
display(pd.DataFrame({"users_course_id": only_in_wk_actions[:20]}))

,metric,value
0,user_lessons_unique_users_course_id,208672
1,wk_users_courses_actions_unique_users_course_id,208565
2,intersection_users_course_id,208552
3,only_in_user_lessons,120
4,only_in_wk_users_courses_actions,13
5,sets_equal,False


Первые 20 users_course_id только в user_lessons:


,users_course_id
0,452078
1,462958
2,464289
3,471244
4,473285
5,473545
6,475205
7,475768
8,477622
9,479756


Первые 20 users_course_id только в wk_users_courses_actions:


,users_course_id
0,499155
1,731628
2,731635
3,732154
4,732518
5,735579
6,744516
7,745218
8,745274
9,746278


In [14]:
# ============================================================
# Восстановление таблицы: user_id | course_id | users_course_id
# Если не удалось разрешить, users_course_id останется NaN
# ============================================================

# 1) локальные копии
uc = dfs["users_courses"][["user_id", "course_id"]].copy()
ul = dfs["user_lessons"][["user_id", "lesson_id", "group_id", "users_course_id"]].copy()

# ------------------------------------------------------------
# Вспомогательные структуры
# ------------------------------------------------------------

# 2) уникальные user_id из users_courses
unique_user_ids = (
    uc["user_id"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

# Курсы каждого пользователя
user_to_courses = (
    uc.groupby("user_id")["course_id"]
    .agg(lambda s: set(s.dropna()))
    .to_dict()
)

# Для каждой пары (lesson_id, group_id) — пользователи этой группы на этом уроке
lesson_group_to_users = (
    ul.groupby(["lesson_id", "group_id"])["user_id"]
    .agg(lambda s: tuple(pd.unique(s.dropna())))
    .to_dict()
)

# Уникальные строки user_lessons, нужные для алгоритма
ul_unique = ul.drop_duplicates(
    subset=["user_id", "lesson_id", "group_id", "users_course_id"]
).copy()

# ------------------------------------------------------------
# 3)-6) Для каждой строки user_lessons строим пересечение курсов
# ------------------------------------------------------------

rows = []

for _, row in ul_unique.iterrows():
    user_id = row["user_id"]
    lesson_id = row["lesson_id"]
    group_id = row["group_id"]
    users_course_id = row["users_course_id"]

    users_in_group = lesson_group_to_users.get((lesson_id, group_id), tuple())

    # Список множеств course_id для пользователей группы
    course_sets = []
    for u in users_in_group:
        if u in user_to_courses:
            course_sets.append(user_to_courses[u])

    # Пересечение курсов внутри группы
    if len(course_sets) == 0:
        course_intersection = set()
    else:
        course_intersection = set.intersection(*course_sets)

    rows.append({
        "user_id": user_id,
        "lesson_id": lesson_id,
        "group_id": group_id,
        "users_course_id": users_course_id,
        "users_in_group": list(users_in_group),
        "courses_intersection_list": sorted(course_intersection),
    })

pair_level_df = pd.DataFrame(rows)

print("pair_level_df:")
display(pair_level_df.head(20))

# ------------------------------------------------------------
# 6) Агрегируем до таблицы user_id | users_course_id | courses_intersection_list
# Для одного users_course_id у пользователя может быть несколько lesson/group,
# поэтому пересекаем их candidate-множества между собой
# ------------------------------------------------------------

def intersect_lists_of_lists(series):
    sets = [set(x) for x in series if isinstance(x, list)]
    if len(sets) == 0:
        return []
    return sorted(set.intersection(*sets))

candidate_df = (
    pair_level_df
    .groupby(["user_id", "users_course_id"], dropna=False)
    .agg(
        courses_intersection_list=("courses_intersection_list", intersect_lists_of_lists),
        lesson_group_count=("lesson_id", "size"),
    )
    .reset_index()
)

candidate_df["candidate_count"] = candidate_df["courses_intersection_list"].apply(len)

print("candidate_df before iterative resolution:")
display(candidate_df.head(20))

# ------------------------------------------------------------
# 7)-8) Итеративное разрешение singleton-кандидатов
# ВАЖНО: исключаем найденные course_id только внутри каждого user_id
# ------------------------------------------------------------

working_df = candidate_df.copy()

resolved_rows = []
resolved_by_user = {}   # user_id -> set(course_id)

changed = True
iteration = 0

while changed:
    changed = False
    iteration += 1

    # Исключаем уже найденные для этого user_id course_id
    new_lists = []
    for _, row in working_df.iterrows():
        user_id = row["user_id"]
        cands = row["courses_intersection_list"]

        already_resolved = resolved_by_user.get(user_id, set())

        if len(cands) <= 1:
            new_lists.append(cands)
        else:
            reduced = [c for c in cands if c not in already_resolved]
            new_lists.append(reduced)

    working_df["courses_intersection_list"] = new_lists
    working_df["candidate_count"] = working_df["courses_intersection_list"].apply(len)

    # Ищем singleton-и
    singleton_df = working_df.loc[working_df["candidate_count"] == 1].copy()

    newly_resolved = 0

    for _, row in singleton_df.iterrows():
        user_id = row["user_id"]
        users_course_id = row["users_course_id"]
        course_id = row["courses_intersection_list"][0]

        # Если этот course_id для данного user_id ещё не фиксировали — фиксируем
        if course_id not in resolved_by_user.get(user_id, set()):
            resolved_by_user.setdefault(user_id, set()).add(course_id)

            resolved_rows.append({
                "user_id": user_id,
                "course_id": course_id,
                "users_course_id": users_course_id
            })

            newly_resolved += 1

    if newly_resolved > 0:
        changed = True

    print(f"Iteration {iteration}: newly_resolved = {newly_resolved}")

resolved_map_df = (
    pd.DataFrame(resolved_rows)
    .drop_duplicates(subset=["user_id", "course_id", "users_course_id"])
    .copy()
)

print("resolved_map_df:")
display(resolved_map_df.head(20))

# ------------------------------------------------------------
# Финальная искомая таблица:
# user_id | course_id | users_course_id
# Если не удалось восстановить, users_course_id = NaN
# ------------------------------------------------------------

result_df = uc.merge(
    resolved_map_df,
    on=["user_id", "course_id"],
    how="left"
)

print("Final result_df:")
display(result_df.head(20))

# ------------------------------------------------------------
# Диагностика
# ------------------------------------------------------------

summary_df = pd.DataFrame([
    {"metric": "users_courses_rows", "value": len(uc)},
    {"metric": "unique_users_courses_pairs", "value": len(uc.drop_duplicates(subset=["user_id", "course_id"]))},
    {"metric": "candidate_rows_user_id_users_course_id", "value": len(candidate_df)},
    {"metric": "resolved_rows_user_id_course_id_users_course_id", "value": len(resolved_map_df)},
    {"metric": "final_rows_with_restored_users_course_id", "value": result_df["users_course_id"].notna().sum()},
    {"metric": "final_share_restored", "value": result_df["users_course_id"].notna().mean()},
])

display(summary_df)

print("Неразрешённые строки:")
display(result_df.loc[result_df["users_course_id"].isna()].head(20))

KeyboardInterrupt: 

In [15]:
# ============================================================
# Эффективное восстановление users_course_id:
# почти всё через merge/groupby, цикл только на финальном разрешении
# ============================================================

uc = dfs["users_courses"][["user_id", "course_id"]].drop_duplicates().copy()
ul = dfs["user_lessons"][["user_id", "lesson_id", "group_id", "users_course_id"]].drop_duplicates().copy()

# ------------------------------------------------------------
# Шаг 1. Для каждого (lesson_id, group_id) собираем пользователей группы
# ------------------------------------------------------------
group_members = ul[["lesson_id", "group_id", "user_id"]].drop_duplicates()

# ------------------------------------------------------------
# Шаг 2. Для каждого пользователя знаем его course_id из users_courses
# ------------------------------------------------------------
user_courses = uc[["user_id", "course_id"]].drop_duplicates()

# ------------------------------------------------------------
# Шаг 3. Для каждой группы получаем все (lesson_id, group_id, group_user_id, course_id)
# ------------------------------------------------------------
group_member_courses = group_members.merge(
    user_courses,
    on="user_id",
    how="left"
)

# Сколько пользователей в каждой группе на уроке
group_sizes = (
    group_members
    .groupby(["lesson_id", "group_id"])["user_id"]
    .nunique()
    .rename("group_size")
    .reset_index()
)

# Для каждой группы и каждого course_id —
# сколько пользователей этой группы имеют этот course_id
group_course_support = (
    group_member_courses
    .groupby(["lesson_id", "group_id", "course_id"])["user_id"]
    .nunique()
    .rename("course_support")
    .reset_index()
)

group_course_support = group_course_support.merge(
    group_sizes,
    on=["lesson_id", "group_id"],
    how="left"
)

# ------------------------------------------------------------
# Шаг 4. Оставляем только курсы, которые есть у ВСЕХ пользователей группы
# Это и есть group intersection, но в табличной форме
# ------------------------------------------------------------
group_course_intersection = group_course_support.loc[
    group_course_support["course_support"] == group_course_support["group_size"],
    ["lesson_id", "group_id", "course_id"]
].copy()

print("group_course_intersection:")
display(group_course_intersection.head(20))

# ------------------------------------------------------------
# Шаг 5. Присоединяем эти candidate course_id к строкам user_lessons
# То есть для каждой (user_id, lesson_id, group_id, users_course_id)
# получаем candidate course_id
# ------------------------------------------------------------
ul_candidates = ul.merge(
    group_course_intersection,
    on=["lesson_id", "group_id"],
    how="left"
)

print("ul_candidates:")
display(ul_candidates.head(20))

# ------------------------------------------------------------
# Шаг 6. Для каждого (user_id, users_course_id) оставляем только те course_id,
# которые встречаются во ВСЕХ его lesson/group внутри этого users_course_id
# Это эквивалент пересечения candidate-множеств по урокам
# ------------------------------------------------------------

# Сколько разных (lesson_id, group_id) у данного (user_id, users_course_id)
user_uc_group_count = (
    ul[["user_id", "users_course_id", "lesson_id", "group_id"]]
    .drop_duplicates()
    .groupby(["user_id", "users_course_id"])
    .size()
    .rename("pair_count")
    .reset_index()
)

# Для каждого (user_id, users_course_id, course_id)
# считаем, в скольких разных (lesson_id, group_id) этот candidate встретился
user_uc_course_support = (
    ul_candidates[["user_id", "users_course_id", "lesson_id", "group_id", "course_id"]]
    .drop_duplicates()
    .groupby(["user_id", "users_course_id", "course_id"])
    .size()
    .rename("candidate_support")
    .reset_index()
)

user_uc_course_support = user_uc_course_support.merge(
    user_uc_group_count,
    on=["user_id", "users_course_id"],
    how="left"
)

# Оставляем только course_id, которые поддержаны всеми lesson/group этой пары
candidate_long = user_uc_course_support.loc[
    user_uc_course_support["candidate_support"] == user_uc_course_support["pair_count"],
    ["user_id", "users_course_id", "course_id"]
].copy()

print("candidate_long:")
display(candidate_long.head(20))

# ------------------------------------------------------------
# Шаг 7. Диагностика до итеративного разрешения
# ------------------------------------------------------------
candidate_count_before = (
    candidate_long
    .groupby(["user_id", "users_course_id"])["course_id"]
    .nunique()
    .rename("candidate_count")
    .reset_index()
)

print("candidate_count_before:")
display(candidate_count_before.head(20))

# ------------------------------------------------------------
# Шаг 8. Итеративное разрешение singleton-кандидатов
# Здесь цикл остаётся, но он уже работает по компактной таблице кандидатов,
# а не по большим исходным таблицам
# ------------------------------------------------------------

working = candidate_long.copy()
resolved = pd.DataFrame(columns=["user_id", "course_id", "users_course_id"])

iteration = 0

while True:
    iteration += 1

    candidate_counts = (
        working
        .groupby(["user_id", "users_course_id"])["course_id"]
        .nunique()
        .rename("candidate_count")
        .reset_index()
    )

    singleton_pairs = candidate_counts.loc[candidate_counts["candidate_count"] == 1].copy()

    if singleton_pairs.empty:
        print(f"Iteration {iteration}: no more singleton pairs")
        break

    singleton_resolved = working.merge(
        singleton_pairs[["user_id", "users_course_id"]],
        on=["user_id", "users_course_id"],
        how="inner"
    ).drop_duplicates()

    before_len = len(resolved)

    resolved = (
        pd.concat([resolved, singleton_resolved[["user_id", "course_id", "users_course_id"]]], ignore_index=True)
        .drop_duplicates()
    )

    if len(resolved) == before_len:
        print(f"Iteration {iteration}: nothing new resolved")
        break

    # Удаляем уже найденные course_id у того же user_id
    working = working.merge(
        resolved[["user_id", "course_id"]].drop_duplicates().assign(_resolved=1),
        on=["user_id", "course_id"],
        how="left"
    )

    # singleton-пары оставляем как есть, а неоднозначные — чистим
    singleton_index = set(
        map(tuple, singleton_pairs[["user_id", "users_course_id"]].drop_duplicates().to_records(index=False))
    )

    working["_pair_key"] = list(zip(working["user_id"], working["users_course_id"]))

    working = working.loc[
        (working["_pair_key"].isin(singleton_index)) |
        (working["_resolved"].isna())
    ].copy()

    working = working.drop(columns=["_resolved", "_pair_key"])

    print(f"Iteration {iteration}: resolved total = {len(resolved)}")

print("resolved:")
display(resolved.head(20))

# ------------------------------------------------------------
# Шаг 9. Финальная таблица user_id | course_id | users_course_id
# Если не разрешилось — NaN
# ------------------------------------------------------------
result_df = uc.merge(
    resolved,
    on=["user_id", "course_id"],
    how="left"
)

summary_df = pd.DataFrame([
    {"metric": "users_courses_pairs", "value": len(uc)},
    {"metric": "resolved_pairs", "value": result_df["users_course_id"].notna().sum()},
    {"metric": "resolved_share", "value": result_df["users_course_id"].notna().mean()},
])

display(summary_df)
display(result_df.head(20))

group_course_intersection:


,lesson_id,group_id,course_id
1,5658,5781,756
9,5659,5782,756
14,5659,5782,50000592
16,5660,5783,757
17,5660,5783,758
18,5668,5791,757
19,5668,5791,758
20,5671,5794,759
22,5672,5795,759
23,5673,5796,759


ul_candidates:


,user_id,lesson_id,group_id,users_course_id,course_id
0,665768,50004620,50004743,449077,50000592
1,665768,50004625,50004748,449077,50000592
2,665768,50004621,50004744,449077,50000592
3,665768,50004622,50004745,449077,50000592
4,665768,50004623,50004746,449077,50000592
5,665828,170005364,170005487,449208,170000688
6,669996,170005364,170005487,457796,170000688
7,665791,50004671,50004794,449128,50000592
8,665810,170005365,170005488,449171,170000688
9,665791,50004607,50004730,449128,50000592


candidate_long:


...

,user_id,users_course_id,candidate_count
0,665768,449077,1
1,665778,449101,1
2,665791,449128,1
3,665795,449138,1
4,665795,449139,1
5,665800,449154,1
6,665805,449161,1
7,665806,449166,1
8,665808,449164,1
9,665810,449171,1


Iteration 1: resolved total = 142871
Iteration 2: resolved total = 150062
Iteration 3: resolved total = 156183
Iteration 4: nothing new resolved
resolved:


,user_id,course_id,users_course_id
0,665768,50000592,449077
1,665778,170000688,449101
2,665791,50000592,449128
3,665795,50000592,449138
4,665795,170000688,449139
5,665800,170000688,449154
6,665805,50000592,449161
7,665806,170000688,449166
8,665808,50000592,449164
9,665810,170000688,449171


,metric,value
0,users_courses_pairs,267206.000000
1,resolved_pairs,156183.000000
2,resolved_share,0.506225


,user_id,course_id,users_course_id
0,718902,771,<NA>
1,708314,763,540016
2,668960,170000688,<NA>
3,668984,170000688,<NA>
4,703125,770,531721
5,711827,770,552543
6,711827,770,596390
7,680366,771,<NA>
8,754592,770,674343
9,754592,770,707768


In [16]:
# Количество кандидатов course_id среди неразрешённых пар

unresolved_candidate_counts = (
    candidate_long
    .merge(
        resolved[["user_id", "users_course_id"]].drop_duplicates().assign(is_resolved=1),
        on=["user_id", "users_course_id"],
        how="left"
    )
    .loc[lambda df: df["is_resolved"].isna()]
    .groupby(["user_id", "users_course_id"])["course_id"]
    .nunique()
    .rename("candidate_course_count")
    .reset_index()
    .sort_values("candidate_course_count", ascending=False)
)

display(unresolved_candidate_counts.head(30))

display(
    unresolved_candidate_counts["candidate_course_count"]
    .value_counts()
    .rename_axis("candidate_course_count")
    .reset_index(name="pair_count")
    .sort_values("candidate_course_count")
)

,user_id,users_course_id,candidate_course_count
10,671550,592404,29
14,671864,518996,15
9,671546,516172,13
8,671546,515048,13
11,671864,518991,13
15,671864,518997,13
7,671546,515043,11
32,696785,519003,11
12,671864,518992,11
13,671864,518994,11


,candidate_course_count,pair_count
1,2,2680
0,3,3974
2,4,1609
5,5,2
3,11,5
4,13,4
7,15,1
6,29,1


In [17]:
# Список course_id, которые чаще всего остались неразрешёнными
# в финальной таблице result_df (где users_course_id = NA)

unresolved_courses_stats = (
    result_df.loc[result_df["users_course_id"].isna()]
    .groupby("course_id")
    .size()
    .rename("na_count")
    .reset_index()
    .sort_values("na_count", ascending=False)
    .reset_index(drop=True)
)

display(unresolved_courses_stats.head(50))

,course_id,na_count
0,771,45061
1,772,41560
2,763,27621
3,836,9708
4,770,6166
5,50000592,4468
6,170000688,2640
7,1061,2517
8,1062,2410
9,1103,2275
